# CHIRPS Precipitation Variability

This script extracts CHIRPS precipitation values using spatial buffers around study sites, computes annual and site-level precipitation variability metrics, and merges them with ET and groundwater datasets for further analysis.

In [12]:
# Import libraries
from collections import defaultdict
import numpy as np
import pandas as pd
import rasterio
from rasterio.mask import mask
from pyproj import Transformer
from shapely.geometry import box
import glob
import re

In [13]:
# Load ET and groundwater merged dataset
# Change file paths ⚠️
et_gw_merged_all_sites = "/capstone/aridgw/outputs/4km/et_precipt_ratio_4km.csv"
et_gw_merged_all_sites = pd.read_csv(et_gw_merged_all_sites)
et_gw_merged_all_sites.head()

,year_value,site_id,depth_to_gw_ft,depth_to_gw_m,latitude,longitude,data_source,region,bbox_side,open_et_version,scaled_annual_et_avg,mean_et,mean_precip,et_precip_ratio,precip_variation_x,precip_variation_y
0,2000,KSGS.371852100505801,239.39,72.966072,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,678.880,702.41219,551.53815,1.273551,1.902489,1.902489
1,2001,KSGS.371852100505801,241.96,73.749408,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,668.881,702.41219,551.53815,1.273551,1.902489,1.902489
2,2002,KSGS.371852100505801,242.78,73.999344,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,639.741,702.41219,551.53815,1.273551,1.902489,1.902489
3,2003,KSGS.371852100505801,246.71,75.197208,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,704.653,702.41219,551.53815,1.273551,1.902489,1.902489
4,2004,KSGS.371852100505801,247.71,75.502008,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,671.125,702.41219,551.53815,1.273551,1.902489,1.902489


In [ ]:
# Define directory containing CHIRPS daily precipitation rasters
data_dir = "/capstone/aridgw/raw_data/chirps_daily_data/"

# Get list of CHIRPS raster files for 2000–2020 period
files = sorted(glob.glob(data_dir + "*.tif"))
files = [f for f in files if any(str(year) in f for year in range(2000, 2021))]

print(len(files), "files found")

In [ ]:
# Extract year from filename using regex pattern matching
def get_year(f):
    return int(re.findall(r"\d{4}", f)[0])

## To change the buffer size, edit `buffer_km=0.5` to half the length of the buffer you want (example: for a 10 km buffer use `buffer_km=5`) in the following line:

`half_size = 0.5 / 111.32`

-   0.5 km → 1 km total buffer
-   1 km → 2 km total buffer
-   2 km → 4 km total buffer
-   5 km → 10 km total buffer

## When changing buffer size, you must also update all related file names and labels, including:
-   Output filenames (e.g., 1km → 2km, 4km, 10km)
-   Variable names that include resolution or scale (e.g., et_precip_ratio_1km)
-   Folder names if used (e.g., /outputs/1km/)

The core computation (masking, raster reading, aggregation) does not need to change. Only the buffer parameter and naming conventions must be updated for consistency

In [ ]:
# Extract year from filename using regex pattern matching
sites = et_gw_merged_all_sites[["site_id", "longitude", "latitude"]].drop_duplicates()

# Initialize dictionary to store precipitation values by site and year
data = defaultdict(list)

# Loop through each CHIRPS raster file
for f in files:
    year = get_year(f)

    with rasterio.open(f) as src:
        transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
        
        # Loop through each site and extract buffered precipitation
        for _, row in sites.iterrows():
            x, y = transformer.transform(row["longitude"], row["latitude"])

            # Create square buffer around site (0.5 km half-width)
            # Change `buffer_km=0.5` ⚠️
            half_size = 2 / 111.32
            geom = box(x - half_size, y - half_size, x + half_size, y + half_size)

            try:
                # Mask raster using buffer geometry and compute mean precipitation
                out, _ = mask(src, [geom], crop=True, all_touched=True)
                arr = out[0].astype(float)
                arr[arr <= 0] = np.nan
                data[(row["site_id"], year)].append(np.nanmean(arr))
            except Exception:
                continue


# Aggregate yearly precipitation statistics per site
records = []

for (site, year), vals in data.items():
    vals = np.array(vals)
    mean = np.nanmean(vals)
    sd = np.nanstd(vals)
    cv = sd / mean if mean > 0 and not np.isnan(mean) else np.nan

    records.append({
        "site_id": site,
        "year_value": year,
        "precip_mean": mean,
        "precip_sd": sd,
        "precip_variation": cv
    })

annual_df = pd.DataFrame(records)

# Collapse annual values into site-level precipitation statistics
precip_variation_df = (
    annual_df
    .groupby("site_id")
    .agg(
        precip_mean=("precip_mean", "mean"),
        precip_variation=("precip_variation", "mean")
    )
    .reset_index()
)

precip_variation_df.head()

In [ ]:
# Merge precipitation variability metrics into main dataset
et_gw_merged_all_sites = et_gw_merged_all_sites.merge(precip_variation_df, on='site_id', how='outer')
et_gw_merged_all_sites

In [ ]:
# Remove redundant precipitation mean column after merging
et_gw_merged_all_sites = et_gw_merged_all_sites.drop(columns=["precip_mean"])

In [23]:
# Save et_gw_merged_all_sites as a csv to outputs folder in remote server
# Change file paths ⚠️
et_gw_merged_all_sites.to_csv(
     "/capstone/aridgw/outputs/4km/et_precipt_ratio_4km.csv",
     index = False)

# Save et_gw_merged_all_sites as a csv to outputs folder locally
# Change file paths ⚠️
et_gw_merged_all_sites.to_csv(
     "../outputs/et_precipt_ratio_4km.csv",
     index = False)

In [24]:
# Reload saved dataset for validation
# Change file paths ⚠️
et_gw_merged_all_sites = "/capstone/aridgw/outputs/4km/et_precipt_ratio_4km.csv"
et_gw_merged_all_sites = pd.read_csv(et_gw_merged_all_sites)
et_gw_merged_all_sites.head()

,year_value,site_id,depth_to_gw_ft,depth_to_gw_m,latitude,longitude,data_source,region,bbox_side,open_et_version,scaled_annual_et_avg,mean_et,mean_precip,et_precip_ratio,precip_variation
0,2000,KSGS.371852100505801,239.39,72.966072,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,678.880,702.41219,551.53815,1.273551,1.902489
1,2001,KSGS.371852100505801,241.96,73.749408,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,668.881,702.41219,551.53815,1.273551,1.902489
2,2002,KSGS.371852100505801,242.78,73.999344,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,639.741,702.41219,551.53815,1.273551,1.902489
3,2003,KSGS.371852100505801,246.71,75.197208,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,704.653,702.41219,551.53815,1.273551,1.902489
4,2004,KSGS.371852100505801,247.71,75.502008,37.31502,-100.8505,USGS,Southern Kansas,4,2.0,671.125,702.41219,551.53815,1.273551,1.902489


In [25]:
# Remove duplicate site entries in precipitation variation table
precip_variation_df = et_gw_merged_all_sites[["site_id", "precip_variation"]].drop_duplicates(subset="site_id")
precip_variation_df

,site_id,precip_variation
0,KSGS.371852100505801,1.902489
21,KSGS.372043101363101,1.910675
42,KSGS.372539100142504,1.759292
63,KSGS.373331098033301,1.872105
84,KSGS.373607100565301,1.870267
105,KSGS.374111099070401,1.869942
126,KSGS.374125100344101,1.732980
147,KSGS.374747100552101,1.820941
168,KSGS.375145100485701,1.867307
189,KSGS.375454101075401,1.836227
